# Lois d’échelle : compute, paramètres et énergie des modèles IA

Exploration des modèles IA (frontier) et du hardware ML pour relier taille de modèle, compute d’entraînement et efficacité énergétique.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (9, 5)

root = Path('data/ai_models')
frontier = pd.read_csv(root / 'frontier_ai_models.csv')

# Nettoyage des colonnes clés
frontier['Parameters'] = pd.to_numeric(frontier['Parameters'], errors='coerce')
frontier['Training compute (FLOP)'] = pd.to_numeric(frontier['Training compute (FLOP)'], errors='coerce')

subset = frontier.dropna(subset=['Parameters', 'Training compute (FLOP)'])
subset = subset[subset['Parameters'] > 0]
subset['log_params'] = np.log10(subset['Parameters'])
subset['log_compute'] = np.log10(subset['Training compute (FLOP)'])

ax = sns.scatterplot(data=subset, x='log_params', y='log_compute', hue=subset['Organization'].fillna('Autre'), alpha=0.7)
ax.set_xlabel('log10(Paramètres)')
ax.set_ylabel('log10(Compute entraînement FLOP)')
ax.set_title('Relation taille de modèle vs compute (frontier models)')
plt.tight_layout()
plt.show()


In [ ]:
# Compute par paramètre (efficience algorithme)
subset['compute_per_param'] = subset['Training compute (FLOP)'] / subset['Parameters']
ax = sns.histplot(subset['compute_per_param'].dropna(), bins=40, log_scale=True, color='teal')
ax.set_xlabel('Compute par paramètre (FLOP/paramètre)')
ax.set_title('Distribution du compute par paramètre')
plt.tight_layout()
plt.show()


In [ ]:
# Hardware : efficacité énergétique
hw = pd.read_csv('data/ml_hardware/ml_hardware.csv')
hw['TDP (W)'] = pd.to_numeric(hw['TDP (W)'], errors='coerce')
hw['Tensor-FP16/BF16 performance (FLOP/s)'] = pd.to_numeric(hw['Tensor-FP16/BF16 performance (FLOP/s)'], errors='coerce')

hw_eff = hw.dropna(subset=['TDP (W)', 'Tensor-FP16/BF16 performance (FLOP/s)']).copy()
hw_eff['TFLOPS_per_W'] = hw_eff['Tensor-FP16/BF16 performance (FLOP/s)'] / 1e12 / hw_eff['TDP (W)']

# On prend les 15 matériels les plus efficaces
best = hw_eff.sort_values('TFLOPS_per_W', ascending=False).head(15)
ax = sns.barplot(data=best, y='Hardware name', x='TFLOPS_per_W', palette='crest')
ax.set_xlabel('TFLOPS FP16 par Watt')
ax.set_ylabel('Matériel')
ax.set_title('Efficacité énergétique (top matériel ML)')
plt.tight_layout()
plt.show()


## Lecture rapide
- Les points log-log montrent la loi d’échelle classique : compute croît plus vite que le nombre de paramètres.
- La distribution du compute/paramètre varie fortement : certaines architectures sont plus efficientes (optimisations algorithmiques, données synthétiques, etc.).
- Côté hardware, le top matériel offre plusieurs dizaines de TFLOPS/W, mais le TDP reste élevé : l’alimentation et le refroidissement deviennent critiques.
- Pour contenir l’“energy wall”, il faut combiner efficacité modèle (moins de FLOP/paramètre) et efficacité matérielle (plus de TFLOPS/W).
